In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel("/content/Monthly_CheckIn_Out_Report_04-03-2026 (1).xlsx")

In [ ]:
emp_cols = ["OrgEmpCode","OrganizationUnit","Designation","Department"]

In [ ]:
date_cols = [c for c in df.columns if "/" in str(c)]

In [ ]:
date_cols

['01/02/2026',
 '02/02/2026',
 '03/02/2026',
 '04/02/2026',
 '05/02/2026',
 '06/02/2026',
 '07/02/2026',
 '08/02/2026',
 '09/02/2026',
 '10/02/2026',
 '11/02/2026',
 '12/02/2026',
 '13/02/2026',
 '14/02/2026',
 '15/02/2026',
 '16/02/2026',
 '17/02/2026',
 '18/02/2026',
 '19/02/2026',
 '20/02/2026',
 '21/02/2026',
 '22/02/2026',
 '23/02/2026',
 '24/02/2026',
 '25/02/2026',
 '26/02/2026',
 '27/02/2026',
 '28/02/2026']

In [ ]:
long_df = df.melt(
    id_vars=emp_cols,
    value_vars=date_cols,
    var_name="Date",
    value_name="Raw_Attendance"
)

In [ ]:
from datetime import datetime

def parse_attendance(val):
    if pd.isna(val):
        return pd.Series([None, None, None, None])

    val = str(val)
    if val in ["AA", "WO"]:
        return pd.Series([val, None, None, None])

    parts = val.split("|")
    status = parts[0]
    in_time, out_time, worked_hrs = None, None, None

    if len(parts) >= 2:
        try:
            time_range = parts[1].split("-")
            if len(time_range) == 2:
                in_time, out_time = time_range[0].strip(), time_range[1].strip()

                # If worked hours (parts[2]) is missing, calculate it
                if len(parts) >= 3:
                    worked_hrs = parts[2][:5]
                elif in_time and out_time:
                    fmt = "%H:%M"
                    t1 = datetime.strptime(in_time, fmt)
                    t2 = datetime.strptime(out_time, fmt)
                    delta = t2 - t1
                    total_mins = int(delta.total_seconds() / 60)
                    if total_mins < 0: # Handle overnight shifts if any
                         total_mins += 1440
                    hh = total_mins // 60
                    mm = total_mins % 60
                    worked_hrs = f"{hh:02d}:{mm:02d}"
        except:
            pass

    return pd.Series([status, in_time, out_time, worked_hrs])

In [ ]:
long_df[["Status", "In_Time", "Out_Time", "Worked_Hrs"]] = long_df["Raw_Attendance"].apply(parse_attendance)
final_df = long_df.drop(columns="Raw_Attendance")
final_df["Date"] = pd.to_datetime(final_df["Date"], format="%d/%m/%Y")
display(final_df.head(5))

,OrgEmpCode,OrganizationUnit,Designation,Department,Date,Status,In_Time,Out_Time,Worked_Hrs
0,TP35052,Jayanagar,Store Executive,STORE EXECUTIVE,2026-02-01,PP,10:00,20:24,10:23
1,TP38110,Hoodi,Store Executive,RETAIL,2026-02-01,PP,10:02,20:00,09:57
2,TP40663,Kondapur,Store Executive,RETAIL,2026-02-01,PP,10:50,21:19,10:28
3,TP33602,Bommasandra,STORE SALE EXECUTIVE,RETAIL,2026-02-01,PP,09:51,20:01,10:09
4,TP42109,Bommasandra,Store Executive,RETAIL,2026-02-01,AA,None,None,None


In [ ]:
final_df.head(5)

,OrgEmpCode,OrganizationUnit,Designation,Department,Date,Status,In_Time,Out_Time,Worked_Hrs
0,TP35052,Jayanagar,Store Executive,STORE EXECUTIVE,2026-02-01,PP,10:00,20:24,10:23
1,TP38110,Hoodi,Store Executive,RETAIL,2026-02-01,PP,10:02,20:00,09:57
2,TP40663,Kondapur,Store Executive,RETAIL,2026-02-01,PP,10:50,21:19,10:28
3,TP33602,Bommasandra,STORE SALE EXECUTIVE,RETAIL,2026-02-01,PP,09:51,20:01,10:09
4,TP42109,Bommasandra,Store Executive,RETAIL,2026-02-01,AA,None,None,None


In [ ]:
def hhmm_to_minutes(hhmm):
    if not isinstance(hhmm, str) or ":" not in hhmm:
        return 0
    h, m = hhmm.split(":")
    return int(h) * 60 + int(m)

def normalize_work_hours_real(actual_minutes,
                              unit=60,
                              grace=5,
                              min_hours=1,
                              max_hours=10):
    """
    Industry-style normalization using bucket + grace.
    """
    # Safety cap (no overtime)
    actual_minutes = min(actual_minutes, max_hours * unit)

    base_bucket = (actual_minutes // unit) * unit
    next_bucket = base_bucket + unit

    # Distance to next bucket
    if next_bucket - actual_minutes <= grace:
        normalized_minutes = next_bucket
    else:
        normalized_minutes = base_bucket

    normalized_hours = normalized_minutes // unit

    # Enforce minimum payable
    if normalized_hours < min_hours:
        return 0

    return min(normalized_hours, max_hours)

def get_normalized_hours(hhmm_str):
    mins = hhmm_to_minutes(hhmm_str)
    return normalize_work_hours_real(mins)

In [ ]:
# Apply the combined logic to the dataframe
final_df["normalized_work_hrs"] = final_df["Worked_Hrs"].apply(get_normalized_hours)
display(final_df.head())

,OrgEmpCode,OrganizationUnit,Designation,Department,Date,Status,In_Time,Out_Time,Worked_Hrs,normalized_work_hrs
0,TP35052,Jayanagar,Store Executive,STORE EXECUTIVE,2026-02-01,PP,10:00,20:24,10:23,10
1,TP38110,Hoodi,Store Executive,RETAIL,2026-02-01,PP,10:02,20:00,09:57,10
2,TP40663,Kondapur,Store Executive,RETAIL,2026-02-01,PP,10:50,21:19,10:28,10
3,TP33602,Bommasandra,STORE SALE EXECUTIVE,RETAIL,2026-02-01,PP,09:51,20:01,10:09,10
4,TP42109,Bommasandra,Store Executive,RETAIL,2026-02-01,AA,None,None,None,0


In [ ]:
final_df.to_excel("Attendnce_Tanka.xlsx")

###extracting the tanka pay done

In [ ]:
spd=pd.read_excel("/content/Attendnce_Tanka.xlsx")

In [ ]:
salarypd=pd.read_excel("/content/Store emp salary 1.xlsx")

In [ ]:
grouped_df = (
    spd.groupby("OrgEmpCode", as_index=False)
       .agg(
           total_worked_hours=("normalized_work_hrs", "sum"),
           daily_worked_list=("normalized_work_hrs", list)
       )
)

In [ ]:
payroll_df = grouped_df.merge(
    salarypd,
    on="OrgEmpCode",
    how="left"
)

In [ ]:
TOTAL_DAYS =int(input("Enter the number of days:--"))

Enter the number of days:--28


In [ ]:
WEEKOFF_AND_PAID_LEAVE =int(input("Enter the number of paid leave:--"))

Enter the number of paid leave:--6


In [ ]:
HOURS_PER_DAY= payroll_df["daily_working_hours"]
Total_hours = TOTAL_DAYS * HOURS_PER_DAY

In [ ]:
payroll_df["total_days"] = TOTAL_DAYS
payroll_df["weekoff_paid_leave"] = WEEKOFF_AND_PAID_LEAVE

In [ ]:
payroll_df["total_expected_hours"] = (    (TOTAL_DAYS - WEEKOFF_AND_PAID_LEAVE)    * payroll_df["daily_working_hours"])

In [ ]:
payroll_df.loc[payroll_df["total_expected_hours"] < 0, "total_expected_hours"] = 0

In [ ]:
payroll_df.columns = payroll_df.columns.str.strip()

payroll_df["salary_hour"] = payroll_df["salary"] / Total_hours

print("Calculated daily_working_hours successfully.")

Calculated daily_working_hours successfully.


In [ ]:
payroll_df.loc[payroll_df["total_expected_hours"] == 0, "daily_working_hours"] = 0

In [ ]:
payroll_df["base_salary"] = (payroll_df["total_worked_hours"] * payroll_df["salary_hour"])

In [ ]:
payroll_df["Paid_Weekend"] = WEEKOFF_AND_PAID_LEAVE * (payroll_df["salary_hour"] * payroll_df["daily_working_hours"])

In [ ]:
def calculate_leverage(row):

    daily_list = row["daily_worked_list"]
    assigned_hours = row["daily_working_hours"]

    # 1. Take only partial days (< assigned and not 0)
    eligible = [h for h in daily_list if 0 < h < assigned_hours]

    if not eligible:
        return pd.Series([0, 0])

    # 2. Sort ascending (worst days first)
    eligible.sort()

    # 3. Take max 3 days
    selected = eligible[:3]

    n = len(selected)

    # 4. Your exact formula
    difference = (n * assigned_hours) - sum(selected)

    # 5. Cap at 3 hours max
    leverage_hours = min(max(difference, 0), 3)

    leverage_amount = leverage_hours * row["salary_hour"]

    return pd.Series([leverage_hours, leverage_amount])


payroll_df[["leverage_hr", "leverage_amount"]] = payroll_df.apply(
    calculate_leverage,
    axis=1
)

In [ ]:
payroll_df["net_salary"] = payroll_df["base_salary"] + payroll_df["Paid_Weekend"] + payroll_df["leverage_amount"]

In [ ]:
import numpy as np

# Ensure columns are stripped of spaces
payroll_df.columns = payroll_df.columns.str.strip()
payroll_df["deduction"] = np.select(
    [
        payroll_df["salary"] <= 21000,
        (payroll_df["salary"] >= 21001) & (payroll_df["salary"] <= 24999),
        payroll_df["salary"] >= 25000
    ],
    [
        (payroll_df["net_salary"] * 0.0075),  # ESI 0.75%
        0,                                 # No deduction
        200                                # Fixed PD
    ],
    default=0
)

In [ ]:
payroll_df["Final_salary"] =( payroll_df["net_salary"] - payroll_df["deduction"].round(2))
display(payroll_df.head())

,OrgEmpCode,total_worked_hours,daily_worked_list,salary,daily_working_hours,total_days,weekoff_paid_leave,total_expected_hours,salary_hour,base_salary,Paid_Weekend,leverage_hr,leverage_amount,net_salary,deduction,Final_salary
0,TP25195,230,"[10, 10, 10, 8, 9, 9, 9, 9, 0, 8, 8, 9, 9, 9, ...",23000.0,10.0,28,6,220.0,82.142857,18892.857143,4928.571429,3.0,246.428571,24067.857143,0.000000,24067.857143
1,TP25199,239,"[0, 0, 10, 10, 10, 9, 10, 10, 10, 10, 10, 10, ...",42000.0,10.0,28,6,220.0,150.000000,35850.000000,9000.000000,3.0,450.000000,45300.000000,200.000000,45100.000000
2,TP25214,248,"[10, 10, 0, 10, 10, 10, 10, 10, 10, 10, 10, 10...",27000.0,10.0,28,6,220.0,96.428571,23914.285714,5785.714286,2.0,192.857143,29892.857143,200.000000,29692.857143
3,TP25812,90,"[10, 10, 10, 10, 10, 10, 10, 10, 10, 0, 0, 0, ...",23000.0,10.0,28,6,220.0,82.142857,7392.857143,4928.571429,0.0,0.000000,12321.428571,0.000000,12321.428571
4,TP25821,250,"[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 0, 10...",18500.0,10.0,28,6,220.0,66.071429,16517.857143,3964.285714,0.0,0.000000,20482.142857,153.616071,20328.522857


In [ ]:
# Define the desired column order
column_order = [
    'OrgEmpCode',
    'salary',
    'total_days',
    'weekoff_paid_leave',
    'daily_working_hours',
    'total_expected_hours',
    'daily_worked_list',
    'total_worked_hours',
    'salary_hour',
    'base_salary',
    'Paid_Weekend',
    'leverage_hr',
    'leverage_amount',
    'deduction',
    'net_salary',
    'Final_salary'

]

# Filter the dataframe to only include these columns in this specific order
# We check if columns exist to avoid errors
existing_cols = [col for col in column_order if col in payroll_df.columns]
payroll_report = payroll_df[existing_cols].copy()

# Display the rearranged dataframe
display(payroll_report.head())
payroll_report.to_excel("payroll_report.xlsx", index=False)

,OrgEmpCode,salary,total_days,weekoff_paid_leave,daily_working_hours,total_expected_hours,daily_worked_list,total_worked_hours,salary_hour,base_salary,Paid_Weekend,leverage_hr,leverage_amount,deduction,net_salary,Final_salary
0,TP25195,23000.0,28,6,10.0,220.0,"[10, 10, 10, 8, 9, 9, 9, 9, 0, 8, 8, 9, 9, 9, ...",230,82.142857,18892.857143,4928.571429,3.0,246.428571,0.000000,24067.857143,24067.857143
1,TP25199,42000.0,28,6,10.0,220.0,"[0, 0, 10, 10, 10, 9, 10, 10, 10, 10, 10, 10, ...",239,150.000000,35850.000000,9000.000000,3.0,450.000000,200.000000,45300.000000,45100.000000
2,TP25214,27000.0,28,6,10.0,220.0,"[10, 10, 0, 10, 10, 10, 10, 10, 10, 10, 10, 10...",248,96.428571,23914.285714,5785.714286,2.0,192.857143,200.000000,29892.857143,29692.857143
3,TP25812,23000.0,28,6,10.0,220.0,"[10, 10, 10, 10, 10, 10, 10, 10, 10, 0, 0, 0, ...",90,82.142857,7392.857143,4928.571429,0.0,0.000000,0.000000,12321.428571,12321.428571
4,TP25821,18500.0,28,6,10.0,220.0,"[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 0, 10...",250,66.071429,16517.857143,3964.285714,0.0,0.000000,153.616071,20482.142857,20328.522857


In [ ]:
payroll_df.to_excel("final_payroll_report.xlsx", index=False)
print("Final payroll report successfully saved to 'final_payroll_report.xlsx'.")

Final payroll report successfully saved to 'final_payroll_report.xlsx'.
